In [1]:
# ============================================================
# STEP 7 — SFT DATA PREPARATION
# ============================================================

from pathlib import Path
import json
import pandas as pd

SPLITS_DIR = Path("data/splits")

sft_train = read_jsonl(SPLITS_DIR / "sft_train.jsonl")
sft_val = read_jsonl(SPLITS_DIR / "sft_val.jsonl")

print(f"SFT train: {len(sft_train):,}")
print(f"SFT val:   {len(sft_val):,}")

NameError: name 'read_jsonl' is not defined

In [ ]:
example = sft_train[0]

print("QUERY:")
print(example["query"])

print("\nTOOLS:")
print(json.dumps(example["tools"], indent=2))

print("\nANSWER:")
print(json.dumps(example["answers"], indent=2))

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

print(tokenizer)

In [ ]:
print("Chat template available:", tokenizer.chat_template is not None)

In [ ]:
SYSTEM_PROMPT = (
    "You are a function-calling assistant. "
    "Select the appropriate tool and provide the required arguments."
)

In [ ]:
def format_answer(answers):
    return json.dumps(
        answers,
        ensure_ascii=False,
        separators=(",", ":")
    )

In [ ]:
print(format_answer(sft_train[0]["answers"]))

In [ ]:
[{"name":"some_tool","arguments":{"x":5}}]

In [ ]:
example["tools"]

In [ ]:
def format_tools(tools):
    return json.dumps(
        tools,
        ensure_ascii=False,
        separators=(",", ":")
    )

In [ ]:
print(format_tools(sft_train[0]["tools"]))

In [ ]:
def build_messages(example):
    user_content = (
        f"User query:\n{example['query']}\n\n"
        f"Available tools:\n{format_tools(example['tools'])}"
    )

    assistant_content = format_answer(example["answers"])

    return [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": user_content
        },
        {
            "role": "assistant",
            "content": assistant_content
        }
    ]

In [ ]:
messages = build_messages(sft_train[0])

for message in messages:
    print(f"\n--- {message['role'].upper()} ---")
    print(message["content"][:2000])

In [ ]:
formatted_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=False
)

print(formatted_text[:4000])

In [ ]:
inference_messages = build_messages(sft_train[0])[:-1]

prompt = tokenizer.apply_chat_template(
    inference_messages,
    tokenize=False,
    add_generation_prompt=True
)

print(prompt[:4000])

In [ ]:
tokens = tokenizer(
    formatted_text,
    add_special_tokens=False
)

print("Token count:", len(tokens["input_ids"]))

In [ ]:
decoded = tokenizer.decode(
    tokens["input_ids"],
    skip_special_tokens=False
)

print(decoded[:4000])

In [ ]:
def get_token_length(example):
    messages = build_messages(example)

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    return len(
        tokenizer(
            text,
            add_special_tokens=False
        )["input_ids"]
    )

In [ ]:
sample_lengths = [
    get_token_length(x)
    for x in sft_train[:1000]
]

pd.Series(sample_lengths).describe(
    percentiles=[0.5, 0.9, 0.95, 0.99]
)

In [ ]:
def format_for_sft(example):
    messages = build_messages(example)

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    return {
        "text": text
    }

In [ ]:
formatted_example = format_for_sft(sft_train[0])

print(formatted_example["text"])

In [ ]:
sft_train_formatted = [
    format_for_sft(example)
    for example in sft_train
]

sft_val_formatted = [
    format_for_sft(example)
    for example in sft_val
]

print(len(sft_train_formatted))
print(len(sft_val_formatted))

In [ ]:
SFT_DIR = Path("data/sft")
SFT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def save_jsonl(path, records):
    with path.open("w", encoding="utf-8") as f:
        for record in records:
            f.write(
                json.dumps(
                    record,
                    ensure_ascii=False
                ) + "\n"
            )

save_jsonl(
    SFT_DIR / "train.jsonl",
    sft_train_formatted
)

save_jsonl(
    SFT_DIR / "val.jsonl",
    sft_val_formatted
)

print("SFT datasets saved.")

In [ ]:
assert len(sft_train_formatted) == len(sft_train)
assert len(sft_val_formatted) == len(sft_val)

assert all(
    isinstance(x["text"], str) and len(x["text"]) > 0
    for x in sft_train_formatted
)

assert all(
    isinstance(x["text"], str) and len(x["text"]) > 0
    for x in sft_val_formatted
)

print("SFT data integrity checks passed ✅")

In [ ]:
for i in range(3):
    print(f"\n{'='*80}")
    print(f"EXAMPLE {i}")
    print(f"{'='*80}")
    print(sft_train_formatted[i]["text"][:3000])